In [14]:
from __future__ import annotations

import asyncio
import json
import logging
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import AsyncIterator

import aiohttp
from bs4 import BeautifulSoup
from tenacity import (
    before_sleep_log,
    retry,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential,
)

logger = logging.getLogger(__name__)

HEADERS = {"User-Agent": "Mozilla/5.0 (research-scraper/1.0; contact: your@email.com)"}
CONCURRENCY = 12
REQUEST_TIMEOUT = 30
RETRY_ATTEMPTS = 4
BASE_URL = "https://www.rba.gov.au"


@dataclass
class Document:
    url: str
    source_type: str
    year: int | None
    title: str
    text: str
    metadata: dict = field(default_factory=dict)


def _abs(href: str) -> str:
    """Converts a relative RBA href to an absolute URL.

    Args:
        href: The href attribute value from an anchor tag.

    Returns:
        Absolute URL string.
    """
    if href.startswith("http"):
        return href
    return BASE_URL + (href if href.startswith("/") else f"/{href}")


def _build_index_jobs(
    years: list[int],
    include_speeches: bool,
    include_smp: bool,
    include_fsr: bool,
) -> list[tuple[str, str, int]]:
    """Builds the full list of index page jobs across all source types.

    Args:
        years: List of years to scrape.
        include_speeches: Whether to include governor speeches.
        include_smp: Whether to include Statements on Monetary Policy.
        include_fsr: Whether to include Financial Stability Reviews.

    Returns:
        List of (index_url, source_type, year) tuples.
    """
    jobs: list[tuple[str, str, int]] = [
        (f"{BASE_URL}/monetary-policy/rba-board-minutes/{yr}/", "minutes", yr)
        for yr in years
    ]
    if include_speeches:
        jobs += [
            (f"{BASE_URL}/speeches/{yr}/", "speech", yr)
            for yr in years
        ]
    if include_smp:
        jobs += [
            (f"{BASE_URL}/publications/smp/{yr}/{yr}{mo}/", "smp", yr)
            for yr in years
            for mo in ("02", "05", "08", "11")
        ]
    if include_fsr:
        jobs += [
            (f"{BASE_URL}/publications/fsr/{yr}/{yr}{mo}/", "fsr", yr)
            for yr in years
            for mo in ("04", "10")
        ]
    return jobs


def _find_document_links(html: str, source_type: str, year: int) -> list[str]:
    """Extracts document URLs from an RBA index page.

    Args:
        html: Raw HTML of the index page.
        source_type: One of 'minutes', 'speech', 'smp', or 'fsr'.
        year: The year being scraped, used to anchor regex patterns.

    Returns:
        Deduplicated list of absolute document URLs.
    """
    patterns = {
        "minutes": re.compile(rf"{year}-\d{{2}}-\d{{2}}\.html"),
        "speech":  re.compile(rf"speeches/{year}/"),
        "smp":     re.compile(r"publications/smp/"),
        "fsr":     re.compile(r"publications/fsr/"),
    }
    pattern = patterns.get(source_type, re.compile(r"\.html$"))
    soup = BeautifulSoup(html, "lxml")
    return list({
        _abs(a["href"])
        for a in soup.find_all("a", href=True)
        if pattern.search(a["href"])
    })


def _extract_text(html: str) -> str:
    """Extracts clean body text from an RBA document page.

    Args:
        html: Raw HTML of the document page.

    Returns:
        Newline-separated paragraph text, filtered to substantive content.
    """
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()
    content = (
        soup.find("div", class_="article-content")
        or soup.find("main")
        or soup.find("div", id="content")
        or soup.body
    )
    if content is None:
        return ""
    return "\n\n".join(
        p.get_text(separator=" ", strip=True)
        for p in content.find_all("p")
        if len(p.get_text(strip=True)) > 40
    )


def _extract_title(html: str) -> str:
    """Extracts the page title from an RBA document page.

    Args:
        html: Raw HTML of the document page.

    Returns:
        Title string from h1 or <title> tag, or 'untitled' if absent.
    """
    soup = BeautifulSoup(html, "lxml")
    h1 = soup.find("h1")
    if h1:
        return h1.get_text(strip=True)
    title = soup.find("title")
    return title.get_text(strip=True) if title else "untitled"


def _make_fetch(session: aiohttp.ClientSession, semaphore: asyncio.Semaphore):
    """Creates a semaphore-bound, retry-decorated async fetch function.

    Args:
        session: Active aiohttp client session.
        semaphore: Concurrency limiter.

    Returns:
        Async callable that fetches a URL and returns its HTML.
    """
    @retry(
        retry=retry_if_exception_type((aiohttp.ClientError, asyncio.TimeoutError)),
        stop=stop_after_attempt(RETRY_ATTEMPTS),
        wait=wait_exponential(multiplier=1, min=2, max=15),
        before_sleep=before_sleep_log(logger, logging.WARNING),
        reraise=True,
    )
    async def _fetch(url: str) -> str:
        async with semaphore:
            async with session.get(url, headers=HEADERS) as response:
                response.raise_for_status()
                return await response.text()
    return _fetch


async def _crawl_index(fetch, index_url: str, source_type: str, year: int) -> list[str]:
    """Fetches an index page and returns all discovered document URLs.

    Args:
        fetch: Retry-decorated async fetch callable.
        index_url: URL of the index page to crawl.
        source_type: Source category label.
        year: Year being crawled.

    Returns:
        List of absolute document URLs found on the index page.
    """
    try:
        html = await fetch(index_url)
        return _find_document_links(html, source_type, year)
    except Exception as exc:
        logger.warning("Index fetch failed %s: %s", index_url, exc)
        return []


async def _scrape_document(fetch, url: str, source_type: str, year: int) -> Document | None:
    """Fetches and parses a single RBA document page.

    Args:
        fetch: Retry-decorated async fetch callable.
        url: URL of the document to scrape.
        source_type: Source category label.
        year: Publication year.

    Returns:
        Parsed Document instance, or None if fetching or parsing fails.
    """
    try:
        html = await fetch(url)
        text = _extract_text(html)
        if not text:
            return None
        return Document(
            url=url,
            source_type=source_type,
            year=year,
            title=_extract_title(html),
            text=text,
        )
    except Exception as exc:
        logger.warning("Document fetch failed %s: %s", url, exc)
        return None


async def scrape_all(
    years: list[int] | None = None,
    include_speeches: bool = True,
    include_smp: bool = True,
    include_fsr: bool = True,
) -> AsyncIterator[Document]:
    """Scrapes all RBA documents for the given years, yielding results as they complete.

    Args:
        years: Years to scrape. Defaults to 2010–2025 inclusive.
        include_speeches: Whether to include governor speeches.
        include_smp: Whether to include Statements on Monetary Policy.
        include_fsr: Whether to include Financial Stability Reviews.

    Yields:
        Document instances as each page is successfully scraped.
    """
    if years is None:
        years = list(range(2010, 2026))

    semaphore = asyncio.Semaphore(CONCURRENCY)
    timeout = aiohttp.ClientTimeout(total=REQUEST_TIMEOUT)
    connector = aiohttp.TCPConnector(limit=CONCURRENCY * 2, keepalive_timeout=60)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        fetch = _make_fetch(session, semaphore)
        index_jobs = _build_index_jobs(years, include_speeches, include_smp, include_fsr)

        index_results = await asyncio.gather(*[
            _crawl_index(fetch, url, source_type, year)
            for url, source_type, year in index_jobs
        ])

        doc_tasks = [
            asyncio.create_task(_scrape_document(fetch, doc_url, source_type, year))
            for (_, source_type, year), doc_urls in zip(index_jobs, index_results)
            for doc_url in doc_urls
        ]

        logger.info("Scraping %d documents", len(doc_tasks))

        for coro in asyncio.as_completed(doc_tasks):
            doc = await coro
            if doc is not None:
                yield doc


def run_and_save(
    output_path: str | Path = "rba_corpus.jsonl",
    years: list[int] | None = None,
    **kwargs,
) -> int:
    """Scrapes all RBA documents and streams results to a JSONL file.

    Args:
        output_path: Destination file path for the JSONL corpus.
        years: Years to scrape. Defaults to 2010–2025 inclusive.
        **kwargs: Additional keyword arguments forwarded to scrape_all.

    Returns:
        Total number of documents successfully saved.
    """
    output_path = Path(output_path)
    count = 0

    async def _run():
        nonlocal count
        with output_path.open("w", encoding="utf-8") as fh:
            async for doc in scrape_all(years=years, **kwargs):
                fh.write(json.dumps({
                    "url": doc.url,
                    "source_type": doc.source_type,
                    "year": doc.year,
                    "title": doc.title,
                    "text": doc.text,
                }) + "\n")
                count += 1
                if count % 10 == 0:
                    logger.info("Saved %d documents...", count)

    asyncio.run(_run())
    return count

In [15]:
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
total = run_and_save(
    output_path="rba_corpus.jsonl",
    years=list(range(2010, 2026)),
    include_speeches=True,
    include_smp=True,
    include_fsr=True,
)
print(f"\nDone. {total} documents saved to rba_corpus.jsonl")

WARNING Retrying __main__._make_fetch.<locals>._fetch in 2.0 seconds as it raised ClientResponseError: 404, message='Not Found', url='https://www.rba.gov.au/publications/smp/2010/201002/'.
WARNING Retrying __main__._make_fetch.<locals>._fetch in 2.0 seconds as it raised ClientResponseError: 404, message='Not Found', url='https://www.rba.gov.au/publications/smp/2010/201005/'.
WARNING Retrying __main__._make_fetch.<locals>._fetch in 2.0 seconds as it raised ClientResponseError: 404, message='Not Found', url='https://www.rba.gov.au/publications/smp/2011/201105/'.
WARNING Retrying __main__._make_fetch.<locals>._fetch in 2.0 seconds as it raised ClientResponseError: 404, message='Not Found', url='https://www.rba.gov.au/publications/smp/2012/201205/'.
WARNING Retrying __main__._make_fetch.<locals>._fetch in 2.0 seconds as it raised ClientResponseError: 404, message='Not Found', url='https://www.rba.gov.au/publications/smp/2011/201108/'.
WARNING Retrying __main__._make_fetch.<locals>._fetch i


Done. 216 documents saved to rba_corpus.jsonl


In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from transformers import pipeline
from .autonotebook import tqdm as notebook_tqdm

class MacroSentimentBuilder:
    """Processes an RBA JSONL corpus into a macroeconomic sentiment index.

    Attributes:
        classifier: HuggingFace sentiment analysis pipeline.
    """

    _DATE_RE = re.compile(r"(\d{4}-\d{2}-\d{2})")
    _MONTH_RE = re.compile(r"/(\d{4})(\d{2})/?$")
    _CHUNK_SIZE = 2000
    _MAX_CHUNKS = 5
    _ROLLING_WINDOW = 3

    def __init__(self, model_name: str = "ProsusAI/finbert", device: int = -1) -> None:
        """Initialises the sentiment pipeline.

        Args:
            model_name: HuggingFace model identifier.
            device: Device index for inference. -1 for CPU, 0+ for GPU.
        """
        self.classifier = pipeline(
            "sentiment-analysis",
            model=model_name,
            device=device,
            truncation=True,
            max_length=512,
        )

    def _extract_date(self, url: str) -> pd.Timestamp:
        """Parses a publication date from an RBA document URL.

        Args:
            url: Document URL, expected to contain a date or year-month segment.

        Returns:
            Parsed Timestamp, or NaT if no date pattern is matched.
        """
        if m := self._DATE_RE.search(url):
            return pd.to_datetime(m.group(1))
        if m := self._MONTH_RE.search(url):
            return pd.to_datetime(f"{m.group(1)}-{m.group(2)}-01")
        return pd.NaT

    def _score_text(self, text: str) -> float:
        """Computes a signed sentiment score for a document.

        Splits the text into fixed-size chunks, runs batch inference, and
        returns the mean signed score across all chunks. Positive labels
        contribute a positive score, negative labels a negative score, and
        neutral labels contribute zero.

        Args:
            text: Raw document text.

        Returns:
            Mean signed sentiment score in [-1, 1], or 0.0 for empty input.
        """
        chunks = [
            text[i: i + self._CHUNK_SIZE]
            for i in range(0, len(text), self._CHUNK_SIZE)
        ][: self._MAX_CHUNKS]

        if not chunks:
            return 0.0

        label_sign = {"positive": 1.0, "negative": -1.0, "neutral": 0.0}
        scores = [
            label_sign.get(result["label"], 0.0) * result["score"]
            for result in self.classifier(chunks)
        ]
        return float(np.mean(scores))

    def _load_records(self, path: Path) -> list[dict[str, Any]]:
        """Streams and scores all valid documents from a JSONL corpus file.

        Args:
            path: Path to the JSONL corpus file.

        Returns:
            List of dicts with 'date', 'source', and 'sentiment' keys.
        """
        records = []
        with path.open("r", encoding="utf-8") as fh:
            for line in fh:
                if not line.strip():
                    continue
                doc = json.loads(line)
                dt = self._extract_date(doc["url"])
                if pd.isna(dt):
                    continue
                records.append({
                    "date": dt,
                    "source": doc["source_type"],
                    "sentiment": self._score_text(doc["text"]),
                })
        return records

    def build_index(self, jsonl_path: str | Path) -> pd.DataFrame:
        """Builds a smoothed daily RBA sentiment index from a JSONL corpus.

        Scores each document, averages by date, and applies a rolling mean
        to produce a smoothed regime signal.

        Args:
            jsonl_path: Path to the JSONL corpus produced by the scraper.

        Returns:
            DataFrame indexed by date with 'sentiment' and 'rba_regime' columns.
        """
        records = self._load_records(Path(jsonl_path))
        df = (
            pd.DataFrame(records)
            .dropna(subset=["date"])
            .groupby("date")["sentiment"]
            .mean()
            .sort_index()
            .to_frame()
        )
        df["rba_regime"] = (
            df["sentiment"]
            .rolling(window=self._ROLLING_WINDOW, min_periods=1)
            .mean()
        )
        return df

/opt/anaconda3/envs/OptionPricer/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_sentiment = MacroSentimentBuilder().build_index("rba_corpus.jsonl")
df_sentiment.to_csv("rba_sentiment.csv")
df_sentiment

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 43215.87it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,sentiment,rba_regime
date,,
2022-02-02,-0.080184,-0.080184
2022-02-11,-0.228616,-0.154400
2022-02-22,0.000000,-0.102934
2022-03-22,-0.106254,-0.111623
2022-05-03,-0.187913,-0.098056
...,...,...
2025-11-20,0.000000,-0.049252
2025-11-26,-0.169301,-0.056434
2025-12-09,-0.076355,-0.081886


In [3]:
df_sentiment

,sentiment,rba_regime
date,,
2022-02-02,-0.080184,-0.080184
2022-02-11,-0.228616,-0.154400
2022-02-22,0.000000,-0.102934
2022-03-22,-0.106254,-0.111623
2022-05-03,-0.187913,-0.098056
...,...,...
2025-11-20,0.000000,-0.049252
2025-11-26,-0.169301,-0.056434
2025-12-09,-0.076355,-0.081886
